# WISER Module 6 — Machine Learning for Addiction Research

Build a prediction prototype, compare models, audit subgroup results, and explain your findings in a model card and one-page advisory-board PDF. This notebook supports **WISER Sections 6.3–6.4**.

Please integrate the use of any AI tools you have to complete this colab.

**Choose one pathway:** Complete **Foundational** or **Applied**, matching your choice in WISER. Follow the corresponding prompts and submission checklist below.

**About this activity:** The supplied `return_to_use_6mo` is an artificial teaching label. The file cannot resolve prescribed-use exclusions, ambiguous kratom findings or missing toxicology. Treat the results as methods practice. Foundational writes a ~200-word fairness reflection; Applied completes the expanded modeling/audit workflow and a ~500-word reflection.

**Use permitted data:** Do not enter PHI, identifiable clinical records or images, restricted participant data, or other sensitive information.


## Start here

1. Open this notebook in Google Colab. If needed, choose **File → Upload notebook** and select the `.ipynb` file.
2. Choose **File → Save a copy in Drive**. Work in that copy.
3. Set `PATHWAY` in the next code cell to `"Foundational"` or `"Applied"`, matching WISER.
4. Work from top to bottom. Click **▶** beside a code cell or press **Shift+Enter** to run it. Read its output before moving on.

**How to enter answers:** Edit the code cell, not the table or message below it. Replace the entire `[bracketed prompt]`, including brackets, with your own answer. Keep quotation marks, field names, commas, and braces. For a table, edit the quoted values in each row of the code; then run it to rebuild the displayed table. Code colors are syntax highlighting, not edit instructions.

**Action labels:** **Run** means no edit is required. **Edit, then run** names the fields to change. **Run, then answer** means inspect the output first and complete the response cell that follows. Complete the work for your pathway; leave the other pathway's answer fields unchanged.

**After a pause:** Save first. When you return, rerun code cells from the top, keeping your answers. If you change an earlier answer or setting, rerun that cell and the cells below it so the saved files match.

**If an error appears:** Read the first error below the cell. Correct the named issue and rerun. If you need help, share that error and its section name with your instructor.

### How this module builds your skills

Build on Module 5: move from describing a pattern to evaluating a prediction. Reuse the checks for outcome definitions, denominators, fair comparisons, and plain-language communication. Now compare models and justify a model and threshold for discussion.

### Your Module 6 checklist

| Stage | What you personally do |
| --- | --- |
| Source and outcome | Foundational: run the synthetic loader. Applied: document and adapt any approved public-use source. Write a proposed six-month clinical outcome and intended use; the provided teaching label cannot implement that definition. |
| Models | Run the preprocessing and comparison cells; read the comparison table. Choose `selected_model` from its exact model names and choose a teaching threshold after examining errors. |
| Fairness and governance | Run the subgroup audit; cite at least two subgroup values **and sample sizes**; explain false-positive/false-negative consequences, the threshold tradeoff and who would review decisions. |
| Your pathway | Foundational: compare logistic regression and decision tree and write **~200 words**. Applied: complete the additional tuned forest, calibration and multi-criterion audit and write **~500 words**. |
| Finish | Complete the model card and plain-language brief fields. Export, inspect the one-page PDF and ZIP, and submit them with your viewable notebook link. |


> **Next code cell:** **Edit, then run:** Set `PATHWAY` to your WISER choice; use the exact value `"Foundational"` or `"Applied"`.


In [ ]:
PATHWAY = "Foundational"  # Change to "Applied" if you selected WISER 6.32.
assert PATHWAY in {"Foundational", "Applied"}
print("Pathway:", PATHWAY)

## 1. Load and inspect the data (WISER 6.4: define and prepare)

Foundational uses the supplied synthetic `hopebridge_ml.csv`. Applied may use disclosed synthetic development data or adapt an approved public-use source; the later checks and models require documented, compatible fields.


**Next code cell — **Foundational — run unchanged. Applied — choose your source, then run:** Keep `APPLIED_SOURCE = "synthetic"` for the disclosed teaching workflow, or upload an approved public-use CSV in Colab’s **Files** panel and paste its copied path. A different source must be adapted to the fields in the next checks and feature cell; changing the path alone is insufficient. Inspect the source message, sample rows, and missing-value table.


In [ ]:
from pathlib import Path
import json, re, shutil, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, confusion_matrix
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_SEED = 42
OUT = Path("module6_outputs")
OUT.mkdir(exist_ok=True)
DATA_URL = "https://raw.githubusercontent.com/mcbacloudlab/WISERTraining-Modules-/main/Module%206/Data/hopebridge_ml.csv"
APPLIED_SOURCE = "synthetic"  # Applied: change to an approved public-use CSV path if appropriate.
if PATHWAY == "Foundational" or APPLIED_SOURCE == "synthetic":
    ml = pd.read_csv(DATA_URL)
    source_name = "WISER synthetic hopebridge_ml.csv"
    source_ref = DATA_URL
    synthetic_used = True
elif APPLIED_SOURCE:
    ml = pd.read_csv(APPLIED_SOURCE)
    source_name = APPLIED_SOURCE
    source_ref = APPLIED_SOURCE
    synthetic_used = False
else:
    raise ValueError("Applied: set APPLIED_SOURCE to 'synthetic' or an approved public-use CSV path.")
print("Loaded", len(ml), "rows and", len(ml.columns), "columns from", source_name)
display(ml.head(4))
display(pd.DataFrame({"missing_n":ml.isna().sum(),"type":ml.dtypes.astype(str)}))

### Data and label checks

The `patient_id` is a **synthetic row key** and is never a predictor. The two 6-month outcome columns must not become predictors of each other. Some fields have missing values; imputation belongs *inside* each model pipeline so test data cannot influence training transformations. Check the label balance and read its definition.


> **Next code cell:** **Run:** Review missingness, duplicate keys and the teaching label. Do not change `patient_id` into a feature.


In [ ]:
label = "return_to_use_6mo"
required = {"patient_id", label, "race_ethnicity", "gender_identity", "svi_quartile", "label_definition"}
if not required.issubset(ml.columns):
    raise ValueError("Missing required teaching fields: " + str(sorted(required-set(ml.columns))))
print("Duplicate patient keys:", int(ml["patient_id"].duplicated().sum()))
if ml["patient_id"].duplicated().any():
    raise ValueError("Resolve duplicate person keys before train/test split to avoid leakage.")
print("Outcome counts:", ml[label].value_counts(dropna=False).to_dict())
print("Outcome definition supplied in file:", ml["label_definition"].dropna().unique().tolist())
if set(ml[label].dropna().unique()) != {0,1}:
    raise ValueError("Teaching outcome must contain both 0 and 1 for this classifier.")

## 2. Refine the proposed outcome (WISER 6.31, task 1)

A clinical outcome definition requires decisions about timing, prescribed medication, ambiguous findings, and missing evidence. The supplied artificial label lacks the evidence to implement that definition; the following models still use the teaching label.


**Next code cell — **Edit, then run:** In `outcome_protocol`, write a short proposed rule covering the index visit, six-month window, nonprescribed-use definition, prescribed medication exclusions, ambiguous/missing findings, and reviewer. State that the supplied label cannot implement that rule. In `intended_use`, describe optional additional peer support and who would review its use. Keep the protocol concise (up to 330 characters for the later brief); document further detail in your reflection if needed.


In [ ]:
outcome_protocol = """[Write a concise proposed clinical outcome, time window, exclusions and adjudication process. State that this file cannot implement it.]"""
intended_use = """[Describe how additional peer-support outreach could be considered without reducing anyone's existing services.]"""
print(outcome_protocol)

## 3. Prepare features and a held-out split (WISER 6.4: prepare)

The baseline features below are plausibly available at the index visit in this *teaching* file. We omit outcome columns, `label_definition`, identifiers, and later referrals/follow-up fields that may reflect actions after the index date. Race and gender are retained **for auditing**, not used directly as predictors here; other features may still serve as proxies. Verify feature timing and appropriateness with stakeholders before any real application.

The split is stratified by the teaching outcome. Every imputer and encoder is learned only from training folds.


**Next code cell — **Run:** Check the printed predictor list and train/test sizes. Confirm that `patient_id` and the outcome are absent from predictors. The code learns missing-value handling and encoding from training data. **Applied with a different source:** Edit the feature lists and document feature timing before running; retain separate audit fields for race, gender, and SVI or adapt the audit consistently to your documented source.


In [ ]:
numeric_features = ["age_at_index", "prior_treatment_episodes", "missed_visit_count_90d",
                    "phq9_score", "gad7_score", "svi_quartile", "housing_insecure",
                    "food_insecure", "transportation_barrier", "currently_receiving_moud",
                    "baseline_positive_toxicology"]
categorical_features = ["insurance_type_index", "dsm5_sud_severity", "primary_substance_category"]
features = numeric_features + categorical_features
missing_features = set(features) - set(ml.columns)
if missing_features: raise ValueError("Map source features before continuing: " + str(sorted(missing_features)))
X, y = ml[features].copy(), ml[label].astype(int)
train_idx, test_idx = train_test_split(np.arange(len(ml)), test_size=.25,
                                     random_state=RANDOM_SEED, stratify=y)
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print("Train:",len(train_idx),"Test:",len(test_idx),"| positives in test:",int(y_test.sum()))
print("Predictors:",features)

def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([("impute",SimpleImputer(strategy="median")),
                          ("scale",StandardScaler())]), numeric_features),
        ("cat", Pipeline([("impute",SimpleImputer(strategy="most_frequent")),
                          ("onehot",OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ])

## 4. Compare models (WISER 6.31 and 6.3)

The Foundational pathway fits **logistic regression and a decision tree**. Applied adds a **random forest** with cross-validated hyperparameter search and calibrates it with training-fold Platt scaling. The held-out test set is used once for reporting, not to tune hyperparameters. AUC measures ranking; Brier score measures probability error (lower is better). Neither is a deployment decision.


**Next code cell — **Run:** Compare the model rows. AUC describes ranking (higher is better); Brier score measures probability error (lower is better). Both describe this sample. Foundational compares logistic regression with the decision tree. Applied also runs the supplied forest tuning and calibration; record which settings were selected and explain the added steps in your reflection. Interpretability and subgroup errors also matter; the table does not choose an approved clinical model.


In [ ]:
models = {
    "Logistic regression": Pipeline([("prep",make_preprocessor()),
                                     ("clf",LogisticRegression(max_iter=1500,random_state=RANDOM_SEED))]),
    "Decision tree": Pipeline([("prep",make_preprocessor()),
                               ("clf",DecisionTreeClassifier(max_depth=4,min_samples_leaf=25,random_state=RANDOM_SEED))]),
}
if PATHWAY == "Applied":
    cv = StratifiedKFold(n_splits=3,shuffle=True,random_state=RANDOM_SEED)
    forest_pipeline = Pipeline([("prep",make_preprocessor()),
                                ("clf",RandomForestClassifier(random_state=RANDOM_SEED,n_jobs=-1))])
    search = GridSearchCV(forest_pipeline,
        {"clf__n_estimators":[100,200],"clf__max_depth":[4,8],"clf__min_samples_leaf":[10,20]},
        scoring="roc_auc",cv=cv,n_jobs=-1)
    search.fit(X_train,y_train)
    models["Random forest (CV tuned)"] = search.best_estimator_
    print("Forest settings selected by training-fold CV:",search.best_params_)
    # Train-fold calibration: fit once here and report on untouched test rows below.
    models["Random forest (CV tuned, Platt calibrated)"] = CalibratedClassifierCV(
        estimator=search.best_estimator_,method="sigmoid",cv=3)

scores_by_model = {}
rows=[]
for name, pipeline in models.items():
    pipeline.fit(X_train,y_train)
    prob = pipeline.predict_proba(X_test)[:,1]
    scores_by_model[name]=prob
    rows.append({"model":name,"test_auc":roc_auc_score(y_test,prob),
                 "test_brier":brier_score_loss(y_test,prob)})
comparison=pd.DataFrame(rows).sort_values("test_brier")
display(comparison.round(3))
print("The numbers describe this synthetic held-out sample only.")

## 5. Evaluate calibration and threshold errors (WISER 6.4: evaluate)

Calibration compares predicted probabilities with observed proportions; small bins can be noisy. Model and threshold choices here are exploratory teaching decisions. The default logistic regression is a simple baseline, and its threshold is not an approved outreach policy.


**Next code cell — **Edit settings, then run:** Copy an exact model name from `comparison` into `selected_model`; keep `THRESHOLD = 0.50` for the first pass. Inspect the metrics and calibration plot. In the response cell that follows, justify the model using AUC, Brier score, interpretability, and one limitation. The diagonal means predicted and observed proportions agree; large departures need investigation. Later threshold comparisons are exploratory, so these test results are not an unbiased final estimate after repeated choices.


In [ ]:
selected_model = "Logistic regression"  # EDIT: choose a model name from the comparison table.
if selected_model not in scores_by_model: raise ValueError("Choose a name from comparison['model'].")
prob = scores_by_model[selected_model]
THRESHOLD = 0.50  # EDIT after examining errors and subgroup results; not a clinical cutoff.
pred = (prob >= THRESHOLD).astype(int)
tn,fp,fn,tp = confusion_matrix(y_test,pred,labels=[0,1]).ravel()
metrics = {"AUC":roc_auc_score(y_test,prob),"Brier":brier_score_loss(y_test,prob),
           "threshold":THRESHOLD,"TN":int(tn),"FP":int(fp),"FN":int(fn),"TP":int(tp),
           "sensitivity":tp/(tp+fn),"specificity":tn/(tn+fp)}
display(pd.Series(metrics).round(3))
obs,predict_mean=calibration_curve(y_test,prob,n_bins=6,strategy="quantile")
fig,ax=plt.subplots(figsize=(5.3,4.5))
ax.plot([0,1],[0,1],"--",color="#6E7880",label="Perfect calibration")
ax.plot(predict_mean,obs,"o-",color="#1B667A",label=selected_model)
ax.set(xlabel="Mean predicted probability",ylabel="Observed proportion",title="Calibration on synthetic test data",xlim=(0,1),ylim=(0,1))
ax.legend(frameon=False); fig.tight_layout()
calibration_path=OUT/"calibration_plot.png"; fig.savefig(calibration_path,dpi=160); plt.show()


**Answer, then run:** Complete `model_choice_response` using the metrics and plot above.


In [ ]:
model_choice_response = """[Explain your selected model using AUC, Brier score, interpretability, and one limitation.]"""
print(model_choice_response)

## 6. Audit subgroup fairness (WISER 6.3, 6.31: race and SVI)

The helper reports race/ethnicity and SVI quartile and includes gender as an additional check. **Demographic parity**: predicted-positive share; **equalized-odds components**: sensitivity (TPR) and false-positive rate (FPR); **coarse calibration-within-group**: mean predicted probability alongside observed prevalence and Brier score. Equal rates or a small gap here would **not** establish overall fairness. Consider group sizes, uncertainty, access to care, and the label's validity.

Do not export person-level predictions; only aggregate metrics leave the notebook.


**Next code cell — **Run:** Read `n` and `positives` before comparing groups. `predicted_positive_rate` is the share flagged; `tpr_sensitivity` is the share of actual positives flagged; `fpr` is the share of actual negatives flagged. Compare race/ethnicity and SVI rows, noting small groups. A small gap on one measure does not establish fairness overall.


In [ ]:
audit = ml.iloc[test_idx][["race_ethnicity","gender_identity","svi_quartile"]].copy().reset_index(drop=True)
audit["actual"] = y_test.to_numpy()
audit["score"] = prob
audit["predicted"] = pred

def fairness_audit(frame, group_col):
    result=[]
    for group,part in frame.groupby(group_col,dropna=False):
        tn,fp,fn,tp=confusion_matrix(part["actual"],part["predicted"],labels=[0,1]).ravel()
        row={"dimension":group_col,"group":str(group),"n":len(part),"positives":int(tp+fn),
             "predicted_positive_rate":part["predicted"].mean(),
             "tpr_sensitivity":tp/(tp+fn) if tp+fn else np.nan,
             "fpr":fp/(fp+tn) if fp+tn else np.nan,
             "mean_predicted_probability":part["score"].mean(),
             "observed_prevalence":part["actual"].mean(),
             "calibration_gap":part["score"].mean()-part["actual"].mean(),
             "brier":brier_score_loss(part["actual"],part["score"]),
             "auc":roc_auc_score(part["actual"],part["score"]) if part["actual"].nunique()==2 else np.nan}
        result.append(row)
    return pd.DataFrame(result)

fairness = pd.concat([fairness_audit(audit,col) for col in
                      ["race_ethnicity","svi_quartile","gender_identity"]],ignore_index=True)
display(fairness.round(3))
print("Smallest subgroup n:",fairness.n.min(),"| interpret sparse groups cautiously.")

### Interpret the audit

Compare at least race/ethnicity and SVI quartile. Group size, uncertainty, access to care, and label validity matter alongside numerical differences; a synthetic-group gap does not establish real-world discrimination.


**Next code cell — **Edit, then run:** In `fairness_response`, cite a race/ethnicity comparison and an SVI comparison, using group names, metric values, and sample sizes from `fairness`. Explain one false-positive and one false-negative consequence for optional outreach, and one question for the advisory board. You now choose the comparisons; do not invent numbers. A short summary will be used in the later brief.


In [ ]:
fairness_response = """[Cite two subgroup values and sample sizes. Explain what a false negative or false positive could mean for optional peer-support outreach.]"""
print(fairness_response)

## 7. Explore thresholds and governance (WISER 6.4: decide)

Changing the threshold changes who is flagged and how many false negatives and false positives occur. The values below are **illustrations**. A clinician and advisory board including people with lived experience would need to weigh consequences and decide any permitted use, monitoring schedule, and **stop condition**. A low score must never remove existing services.


**Next code cell — **Run, then answer:** Compare the 0.30, 0.50, and 0.70 rows. In `threshold_response`, choose a threshold for discussion and cite its false-positive/false-negative tradeoff. In `governance_plan`, name reviewer roles, review frequency, whose feedback is needed, and a specific reason to pause use. Rerun this cell after writing. This comparison does not change `THRESHOLD`; if you change it in Section 5, rerun Sections 5 onward to keep the audit and exports consistent.


In [ ]:
threshold_rows=[]
for threshold in [0.30,0.50,0.70]:
    choices=(prob>=threshold).astype(int)
    tn_,fp_,fn_,tp_=confusion_matrix(y_test,choices,labels=[0,1]).ravel()
    threshold_rows.append({"threshold":threshold,"flagged":int(choices.sum()),
                           "false_negatives":int(fn_),"false_positives":int(fp_),
                           "sensitivity":tp_/(tp_+fn_),"specificity":tn_/(tn_+fp_)})
thresholds=pd.DataFrame(threshold_rows)
display(thresholds.round(3))
threshold_response = """[Recommend a threshold for discussion; explain which error tradeoff matters and why this is not an authorized clinical cutoff.]"""
governance_plan = """[Name oversight owners, a review frequency, feedback from peer-support staff and people with lived experience, and a specific condition to pause use.]"""
print(threshold_response)

## 8. Complete the model card and advisory-board brief (WISER 6.4: communicate)

A model card documents purpose, data, outcome, measured performance, limitations, and governance. The advisory brief translates that evidence for a decision-maker, using the values measured here rather than fictional WISER scenario metrics.


**Next code cell — **Edit, then run:** Fill `card_data`, `card_performance`, `card_fairness`, and `card_limitations` from this run. `card_intended_use`, `card_outcome`, and `card_governance` reuse earlier answers. In `brief_finding`, translate one measured result for a nontechnical reader; in `brief_action`, recommend the next evaluation/review step. Keep `card_fairness` ≤300, `card_limitations` ≤320, `brief_finding` ≤250, and `brief_action` ≤260 characters; the PDF cell also checks the reused protocol/governance fields.


In [ ]:
card_intended_use = intended_use
card_outcome = outcome_protocol
card_data = """[Describe the synthetic sample, 6-month label, features, train/test split, missing-value handling, and exclusions.]"""
card_performance = """[Report the selected model's test AUC, Brier score, threshold and TN/FP/FN/TP from this run.]"""
card_fairness = """[Summarize at least two group dimensions and their sizes; note the audit's limits.]"""
card_limitations = """[Explain the artificial label, missingness, potential feature timing/proxy issues, and limited generalizability.]"""
card_governance = governance_plan
brief_finding = """[Explain the selected model's main result in plain language; include a supported number.]"""
brief_action = """[Recommend a next evaluation or stakeholder decision, not deployment from this synthetic file.]"""
model_card = {"intended_use":card_intended_use,"data_and_outcome":card_data+"\n"+card_outcome,
              "performance":card_performance,"fairness":card_fairness,
              "limitations":card_limitations,"governance":card_governance}
print("Model card sections:",list(model_card))

## 9. One-page advisory-board PDF

The generated PDF contains the outcome, selected model, metrics, key fairness finding, threshold, limitations, and safeguards. Complete the text fields above first; open the PDF after export to check readability. Text limits keep the one-page layout from overlapping.


> **Next code cell:** **Run:** Create the one-page advisory-board PDF. If a field is too long, shorten it above and rerun.


In [ ]:
brief_fields = {"outcome":card_outcome,"finding":brief_finding,"fairness":card_fairness,
                "limitations":card_limitations,"action":brief_action,"governance":card_governance}
limits={"outcome":330,"finding":250,"fairness":300,"limitations":320,"action":260,"governance":280}
too_long=[key for key,value in brief_fields.items() if len(value)>limits[key]]
if too_long: raise ValueError("Shorten these brief fields: "+", ".join(too_long))
brief_fig=plt.figure(figsize=(8.5,11),facecolor="white")
brief_fig.text(.075,.966,"HopeBridge | Model advisory brief",fontsize=15,weight="bold",color="#164C5B")
brief_fig.text(.075,.937,"Teaching prototype • Synthetic data • Advisory review required",fontsize=9)

def brief_block(label,value,y,wrap=92):
    lines=textwrap.wrap(value,wrap)
    brief_fig.text(.075,y,label,fontsize=10.5,weight="bold",va="top")
    brief_fig.text(.075,y-.023,"\n".join(lines),fontsize=9,va="top",linespacing=1.28)
    return y-.023-.019*len(lines)-.022

y=.906
for section_title,key in [("Outcome and exclusions","outcome"),("Selected model and finding","finding"),
                  ("Fairness audit","fairness"),("Limits","limitations"),
                  ("Threshold and recommended next step","action"),("Governance and stop condition","governance")]:
    y=brief_block(section_title,brief_fields[key],y)
if y < .08: raise ValueError("Brief text is too long for one page; shorten the response fields.")
brief_fig.text(.075,.047,f"Test AUC {metrics['AUC']:.2f}  |  Brier {metrics['Brier']:.2f}  |  Threshold {THRESHOLD:.2f}",fontsize=8)
brief_fig.text(.075,.028,"Source: WISER synthetic hopebridge_ml.csv; artificial teaching label. No clinical use.",fontsize=7.5)
brief_path=OUT/"Module_6_advisory_board_brief.pdf"
with PdfPages(brief_path) as pdf:pdf.savefig(brief_fig)
plt.close(brief_fig)
print("Saved:",brief_path)

## 10. Reflection and FAIR-aligned export

**Foundational:** Write ~200 words about **one fairness tradeoff**, referencing the subgroup audit and what the advisory board should decide. Submit the completed notebook link and one-page PDF brief.

**Applied:** Write ~500 words about outcome-definition and fairness tradeoffs. Document your public-use source or disclose the synthetic offline-development fallback; include your full modeling, calibration, fairness workflow, model card, and PDF. The outputs ZIP provides files for review. It does not replace the Colab link or the brief PDF.


> **Next code cell:** **Edit, then run:** Replace `reflection` with ~200 words Foundational or ~500 words Applied.


In [ ]:
reflection = """[Write your pathway's reflection: ~200 words Foundational or ~500 words Applied.]"""
print("Words:",len(reflection.split()),"| pathway:",PATHWAY)

> **Next code cell:** **Run last:** Clear any remaining prompts, inspect the card and PDF, and download the documented outputs ZIP.


In [ ]:
responses={"outcome protocol":outcome_protocol,"intended use":intended_use,
           "model choice":model_choice_response,"fairness interpretation":fairness_response,
           "threshold decision":threshold_response,"governance":governance_plan,
           "model card data":card_data,"model card performance":card_performance,
           "model card fairness":card_fairness,"model card limitations":card_limitations,
           "brief finding":brief_finding,"brief next step":brief_action,"reflection":reflection}
unfinished=[key for key,value in responses.items() if "[" in value and "]" in value]
if unfinished:print("TO COMPLETE before submission:",", ".join(unfinished))
low,high=(170,230) if PATHWAY=="Foundational" else (450,550)
if not low<=len(reflection.split())<=high:print(f"TO COMPLETE: write approximately {200 if PATHWAY=='Foundational' else 500} reflection words.")
if PATHWAY=="Applied" and synthetic_used:print("APPLIED: synthetic data were used for offline development; disclose this limitation.")

comparison.to_csv(OUT/"model_comparison.csv",index=False)
fairness.to_csv(OUT/"aggregate_fairness_audit.csv",index=False)
thresholds.to_csv(OUT/"threshold_comparison.csv",index=False)
(OUT/"model_card.md").write_text("# WISER Module 6 model card\n\n"+
    "\n\n".join(f"## {key.replace('_',' ').title()}\n{value}" for key,value in model_card.items())+"\n",encoding="utf-8")
(OUT/"responses_and_reflection.md").write_text("# Learner responses\n\n"+
    "\n\n".join(f"## {key.title()}\n{value}" for key,value in responses.items())+"\n",encoding="utf-8")
metadata={"module":"WISER 6","pathway":PATHWAY,"source":source_name,"source_reference":source_ref,
          "synthetic_used":synthetic_used,"outcome_field":label,"label_note":"Synthetic teaching label, not a clinical definition",
          "selected_model":selected_model,"test_rows":len(test_idx),"threshold":THRESHOLD,
          "feature_columns":features,"audit_dimensions":["race_ethnicity","svi_quartile","gender_identity"],
          "random_seed":RANDOM_SEED,"person_level_predictions_exported":False}
(OUT/"metadata.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
(OUT/"README.md").write_text(
    f"# WISER Module 6 outputs\n\nPathway: {PATHWAY}. Input: {source_ref}.\n\n"
    "Files: model_comparison.csv, aggregate_fairness_audit.csv, threshold_comparison.csv, "
    "calibration_plot.png, model_card.md, responses_and_reflection.md, metadata.json, advisory-board brief PDF.\n\n"
    "All outputs are for teaching. The synthetic outcome has no adjudicated clinical definition. "
    "Only aggregate audit metrics are exported; no patient-level predictions are included. "
    "See the notebook for preprocessing, data split, models and limitations.\n",encoding="utf-8")
archive=shutil.make_archive("WISER_Module_6_outputs","zip",root_dir=".",base_dir=str(OUT))
print("PDF:",brief_path,"| ZIP:",archive)
print("Files:",sorted(p.name for p in OUT.iterdir()))
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Outside Colab: download the ZIP from the file browser.")